# EDA — Penguins (ONG)

Análise exploratória usando `polars` (manipulação), `duckdb` (consultas SQL eficientes) e `seaborn`/`matplotlib` (visualizações).

Objetivos:
- Identificar registros com anotações faltantes.
- Mostrar de quais ilhas a maioria dos pinguins vem.
- Mostrar quais espécies são mais comuns.
- Explorar relações entre medidas e espécie.
- Explorar diferenças entre sexos dentro de cada espécie.

## Setup
Instale as dependências (no venv do projeto):

```
pip install -r requirements.txt
```

In [ ]:
# Imports e estilo
import polars as pl
import duckdb
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

sns.set_theme(style="whitegrid")
penguin_palette = ["#0A2342", "#67B7D8", "#B9E6F2", "#2A2A2A"]
sns.set_palette(penguin_palette)

In [ ]:
# Carregar dados com polars (tratando 'NA' como nulo)
df = pl.read_csv('dataset/penguins.csv', null_values=['NA','NaN',''])
print('Colunas originais:', df.columns)
df.head(5).to_pandas()

In [ ]:
# Renomear colunas para nomes canônicos (em inglês)
cols = df.columns
mapping = {
    cols[0]: 'species',
    cols[1]: 'island',
    cols[2]: 'bill_length_mm',
    cols[3]: 'bill_depth_mm',
    cols[4]: 'flipper_length_mm',
    cols[5]: 'body_mass_g',
    cols[6]: 'sex'
}
df = df.rename(mapping)
# Forçar tipos numéricos onde aplicável
df = df.with_columns([
    pl.col('bill_length_mm').cast(pl.Float64),
    pl.col('bill_depth_mm').cast(pl.Float64),
    pl.col('flipper_length_mm').cast(pl.Float64),
    pl.col('body_mass_g').cast(pl.Float64)
])
df.head(5).to_pandas()

In [ ]:
# Conectar ao DuckDB em memória e registrar o DataFrame (arrow) para consultas SQL eficientes
con = duckdb.connect()
con.register('penguins', df.to_arrow())
print('Tabelas registradas em DuckDB:', con.execute('SHOW TABLES').fetchall())

## 1) Quais pinguins não têm anotações?
Vamos usar `duckdb` para contar valores faltantes por coluna e listar registros com pelo menos um valor ausente.

In [ ]:
# Contagem de ausências por coluna (uma linha com todas as colunas como colunas separadas)
missing_sql = '''
SELECT
  (COUNT(*) - COUNT(species)) AS species_missing,
  (COUNT(*) - COUNT(island)) AS island_missing,
  (COUNT(*) - COUNT(bill_length_mm)) AS bill_length_missing,
  (COUNT(*) - COUNT(bill_depth_mm)) AS bill_depth_missing,
  (COUNT(*) - COUNT(flipper_length_mm)) AS flipper_length_missing,
  (COUNT(*) - COUNT(body_mass_g)) AS body_mass_missing,
  (COUNT(*) - COUNT(sex)) AS sex_missing
FROM penguins;
'''
missing_row = con.execute(missing_sql).fetchdf()
# Transformar em formato longo para exibição
missing_pd = missing_row.melt(var_name='column', value_name='missing_count').sort_values('missing_count', ascending=False)
missing_pd

In [ ]:
# Mostrar os registros que têm pelo menos um valor ausente (até 20)
missing_rows = con.execute(
    "SELECT * FROM penguins WHERE species IS NULL OR island IS NULL OR bill_length_mm IS NULL OR bill_depth_mm IS NULL OR flipper_length_mm IS NULL OR body_mass_g IS NULL OR sex IS NULL LIMIT 20"
).fetchdf()
missing_rows

## 2) Quais ilhas a maioria dos pinguins está vindo?
Contagem por ilha (usando DuckDB) e visualização.

In [ ]:
island_counts = con.execute(
    "SELECT island, COUNT(*) AS n FROM penguins GROUP BY island ORDER BY n DESC"
).fetchdf()
island_counts

In [ ]:
plt.figure(figsize=(7,4))
sns.barplot(data=island_counts, x='island', y='n', palette=penguin_palette)
plt.title('Número de pinguins por ilha')
plt.ylabel('Contagem')
plt.xlabel('Ilha')
plt.tight_layout()
plt.show()

## 3) Quais as espécies que a ONG mais possui?
Contagem por espécie e gráfico.

In [ ]:
species_counts = con.execute(
    "SELECT species, COUNT(*) AS n FROM penguins GROUP BY species ORDER BY n DESC"
).fetchdf()
species_counts

In [ ]:
plt.figure(figsize=(7,4))
sns.barplot(data=species_counts, x='species', y='n', palette=penguin_palette)
plt.title('Número de pinguins por espécie')
plt.ylabel('Contagem')
plt.xlabel('Espécie')
plt.tight_layout()
plt.show()

## 4) Existe alguma relação entre as medidas do pinguim e a sua espécie?
Mostraremos estatísticas agregadas (DuckDB) e um `pairplot` (Seaborn) para visualizar separações entre espécies.

In [ ]:
# Estatísticas médias por espécie (DuckDB)
species_stats = con.execute(
    "SELECT species, AVG(bill_length_mm) AS mean_bill_length, AVG(bill_depth_mm) AS mean_bill_depth, AVG(flipper_length_mm) AS mean_flipper_length, AVG(body_mass_g) AS mean_body_mass, COUNT(*) AS n FROM penguins GROUP BY species ORDER BY n DESC"
).fetchdf()
species_stats

In [ ]:
# Pairplot: converter para pandas (Seaborn espera pandas) após limpar NA nas medidas
plot_cols = ['species','bill_length_mm','bill_depth_mm','flipper_length_mm','body_mass_g']
df_plot = df.select(plot_cols).drop_nulls().to_pandas()
sns.pairplot(df_plot, hue='species', palette=penguin_palette, diag_kind='kde', plot_kws={'alpha':0.6})
plt.suptitle('Pairplot: medidas por espécie', y=1.02)
plt.show()

## 5) Existe alguma relação entre as medidas do pinguim e seu sexo para cada espécie?
Faremos boxplots das medidas separadas por `sex`, dentro de cada espécie, e calcularemos médias por grupo (DuckDB).

In [ ]:
measurements = ['bill_length_mm','bill_depth_mm','flipper_length_mm','body_mass_g']
species_list = df_plot['species'].unique()
for sp in species_list:
    sub = df_plot[df_plot['species']==sp].dropna(subset=['sex'])
    fig, axes = plt.subplots(2,2, figsize=(12,8))
    for ax, meas in zip(axes.flatten(), measurements):
        sns.boxplot(data=sub, x='sex', y=meas, ax=ax, palette=penguin_palette[:2])
        ax.set_title(f'{sp} — {meas}')
    fig.suptitle(f'{sp} — Medidas por sexo', y=1.02)
    plt.tight_layout()
    plt.show()

In [ ]:
# Médias por espécie e sexo (DuckDB)
sex_stats = con.execute(
    "SELECT species, sex, AVG(bill_length_mm) AS mean_bill_length, AVG(bill_depth_mm) AS mean_bill_depth, AVG(flipper_length_mm) AS mean_flipper_length, AVG(body_mass_g) AS mean_body_mass, COUNT(*) AS n FROM penguins GROUP BY species, sex ORDER BY species"
).fetchdf()
sex_stats

## Resumo e próximos passos
- As consultas e gráficos acima respondem às perguntas solicitadas: registros com dados faltantes, ilhas de origem mais comuns, espécies mais frequentes, relação entre medidas e espécie, e diferenças entre sexos por espécie.
- Próximos passos sugeridos: testes estatísticos formais (t-test/ANOVA), modelos de classificação (por exemplo, usar `duckdb` para treinar modelos via integração), ou clustering.